# Qwen2.5-1.5B QLoRA 微调 — Google Colab 版
---
**功能**: 在自定义 JSONL 数据上对 Qwen2.5-1.5B 进行 QLoRA 微调
**硬件要求**: Colab T4 GPU (免费版 15GB VRAM 完全够用)
**输出**: LoRA adapter 保存到 Google Drive

---

## 第 1 步: 检查 GPU 环境
确认 Colab 分配到了 GPU（T4 或更好）。

In [ ]:
!nvidia-smi

## 第 2 步: 挂载 Google Drive
训练数据 `wuyin_train.jsonl` 需要提前上传到你的 Google Drive 根目录（或指定路径）。
运行下面代码后会弹出授权链接，点击允许即可。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 检查数据文件是否存在（假设放在 Drive 根目录）
import os
data_path = '/content/drive/MyDrive/wuyin_train.jsonl'
if os.path.exists(data_path):
    print(f'[OK] 数据文件已找到: {data_path}')
else:
    print(f'[WARN] 未找到数据文件，请将 wuyin_train.jsonl 上传到 Google Drive 根目录')

## 第 3 步: 安装依赖
安装所有需要的 Python 包（约 2-3 分钟）。

In [ ]:
!pip install -q torch transformers datasets accelerate peft bitsandbytes trl ninja packaging
!pip install -q flash-attn --no-build-isolation  # 可选，加速训练

print('[OK] 依赖安装完成')

## 第 4 步: 导入库和配置参数
设置所有超参数，可根据需要调整。

In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from trl import SFTTrainer

# ── 配置参数 ──────────────────────────────────────
MODEL_ID = "Qwen/Qwen2.5-1.5B"
DATASET_PATH = "/content/drive/MyDrive/wuyin_train.jsonl"
OUTPUT_DIR = "/content/drive/MyDrive/wuyin-lora-adapter"  # 保存到 Drive
NUM_EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
MAX_SEQ_LENGTH = 2048
LEARNING_RATE = 2e-4
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

print(f'[配置] 模型: {MODEL_ID}')
print(f'[配置] 数据: {DATASET_PATH}')
print(f'[配置] 轮数: {NUM_EPOCHS} | 批大小: {BATCH_SIZE} | 梯度累积: {GRAD_ACCUM_STEPS}')
print(f'[配置] LoRA r={LORA_R} alpha={LORA_ALPHA}')

## 第 5 步: 加载 4-bit 量化模型
使用 bitsandbytes 将 1.5B 模型量化为 4-bit，大幅降低显存占用。
原始 1.5B 模型约需 3GB 显存（FP16），4-bit 量化后仅约 1GB。

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f'[1/5] 加载基础模型: {MODEL_ID} ...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

print(f'[2/5] 加载分词器 ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

## 第 6 步: 配置 LoRA
LoRA 只在注意力层的 Q/V/K/O 投影上添加低秩适配器，
可训练参数仅约 0.2% 的模型总量，训练极快。

In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 第 7 步: 加载和格式化数据
数据格式: 每行一个 JSON 对象，包含 `messages` 列表。

```json
{"messages": [{"role": "user", "content": "你好"}, {"role": "assistant", "content": "你好！有什么可以帮你的？"}]}
```

In [ ]:
print(f'[3/5] 加载数据集: {DATASET_PATH}')
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
print(f'数据集样本数: {len(dataset)}')

def format_chat(example):
    """将 messages 列表转换为 Qwen2.5 chat template 字符串"""
    messages = example["messages"]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = dataset.map(format_chat, remove_columns=dataset.column_names)
print(f'[OK] 数据格式化完成，示例长度: {len(dataset[0]["text"])} 字符')

## 第 8 步: 开始训练
3 个 epoch 在 T4 GPU 上约需 20-60 分钟（取决于数据量）。
每 10 步输出一次 loss，每个 epoch 结束自动保存 checkpoint。

In [ ]:
training_args = TrainingArguments(
    output_dir="/content/wuyin-checkpoints",  # 临时 checkpoint
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    dataloader_num_workers=2,
    report_to="none",
    remove_unused_columns=False,
    seed=42,
)

print(f'[4/5] 开始训练 ({NUM_EPOCHS} epochs) ...')
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
)
trainer.train()
print('[OK] 训练完成!')

## 第 9 步: 保存 LoRA Adapter 到 Google Drive
训练好的 adapter 保存到 Google Drive，不会被 Colab 会话删除。
之后可以在本地或 Colab 中加载使用。

In [ ]:
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'[5/5] 保存 LoRA adapter 到: {OUTPUT_DIR}')
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# 验证保存的文件
saved_files = os.listdir(OUTPUT_DIR)
print(f'保存的文件: {saved_files}')
print(f'[OK] Adapter 已保存到 Google Drive!')

## 第 10 步 (可选): 快速推理测试
加载刚保存的 adapter，测试模型是否能正常生成回复。

In [ ]:
from peft import PeftModel

# 重新加载基础模型 + adapter
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR, trust_remote_code=True)

# 测试推理
test_prompt = "你好，请简单介绍一下你自己。"
messages = [{"role": "user", "content": test_prompt}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.7,
    do_sample=True,
)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f'模型回复:\n{response}')

---
## 训练完成!

你的 LoRA adapter 已保存到 Google Drive: `MyDrive/wuyin-lora-adapter/`

### 下一步:
- **合并模型**: 使用 `merge_and_unload()` 将 adapter 合并到基础模型
- **部署**: 合并后的模型可部署到 Hugging Face / Vercel / 本地推理
- **继续训练**: 加载 adapter checkpoint 继续训练更多 epoch

### 常见问题:
- 如果 OOM: 减小 `BATCH_SIZE` 到 1，增大 `GRAD_ACCUM_STEPS` 到 8
- 如果 Drive 空间不足: 先清理 Drive 或更换 `OUTPUT_DIR` 路径
- 如果训练 loss 不下降: 尝试增大 `LEARNING_RATE` 或检查数据格式
---